In [161]:
!git clone https://github.com/aimacode/aima-python.git

Cloning into 'aima-python'...
remote: Enumerating objects: 5095, done.
remote: Total 5095 (delta 0), reused 0 (delta 0), pack-reused 5095 (from 1)
Receiving objects: 100% (5095/5095), 17.44 MiB | 26.11 MiB/s, done.
Resolving deltas: 100% (3418/3418), done.


In [162]:
%cd aima-python

/content/aima-python/aima-python/aima-python


In [163]:
!pip install -r requirements.txt

In [164]:
from agents import *
from search import *
from utils import *
import random

In [165]:
from copy import deepcopy

In [166]:
"""REFAZENDO: CONSISTÊNCIA DOS ÍNDICES E ROTAÇÃO"""

'REFAZENDO: CONSISTÊNCIA DOS ÍNDICES E ROTAÇÃO'

In [167]:
#w is white, y is yellow, r is red, o is orange, b is blue, g is green
#sides, with white down and blue in the front
#from the perspective where the side is on the front, index 0 is bottom left, index 1 is top left, index 2 is top right, and index 3 is bottom right.
#wait! this perspective isn't unique! trouble!
#for blue you used the perspective with white below
#you're better off making a cube class so you can figure out the permutation maps for the indices that respect the geometry of the cube

In [168]:
DOWN=["w"]*4
LEFT=["o"]*4
BACK=["g"]*4
UP=["y"]*4
RIGHT=["r"]*4
FRONT=["b"]*4

In [169]:
#the cube is simply the concatenation of all 6 lists. indices 0 to 3 are down, 4 to 7 are left, and so on

In [170]:
ESTADO_INICIAL=DOWN+LEFT+BACK+UP+RIGHT+FRONT

In [171]:
#for the canonical cube rotation list, we use the indices

In [172]:
CANONICAL_CUBE=list(range(24))

In [173]:
SIDES=["DOWN","LEFT","BACK","UP","RIGHT","FRONT"]
DIRECTIONS=["CW","CCW"]

In [174]:
#now, we will make turns centered on a face. we will first make only clockwise turns. counterclockwise turns can be obtained as inverses.
#we need to find the stickers adjacent to a face and rotate them. there are 8 adjacent stickers to each face
#as a simplification, we can group adjacent stickers into corners.
#bottom-left sticker of a face is adjacent to top-left of face to its bottom and bottom-right of face to its right.
#more generally, x-y sticker of a face is adjacent to opposite(x)-y sticker of face to its x and x-opposite(y) sticker of face to its y
#problem: need consistent perspective

In [175]:
"""NOTE: ROTATE_CLOCKWISE AND ROTATE_COUNTERCLOCKWISE HAVE OPPOSITE ROTATIONS FOR RED, BLUE AND YELLOW SIDES. TAKE THIS INTO ACCOUNT"""

'NOTE: ROTATE_CLOCKWISE AND ROTATE_COUNTERCLOCKWISE HAVE OPPOSITE ROTATIONS FOR RED, BLUE AND YELLOW SIDES. TAKE THIS INTO ACCOUNT'

In [176]:
def rotate_clockwise(side):
  #ROTATES CLOCKWISE FOR WHITE,GREEN,ORANGE.
  #COUNTERCLOCKWISE FOR YELLOW, BLUE, RED
  return [side[3],side[0],side[1],side[2]]

In [177]:
def rotate_counterclockwise(side):
  #ROTATES COUNTERCLOCKWISE FOR WHITE,GREEN,ORANGE.
  #CLOCKWISE FOR YELLOW, BLUE, RED
  return [side[1],side[2],side[3],side[0]]

In [178]:
#now, the perspective from a clockwise rotation (not turn) of the cube, looking at a side, is that that side (front) gets rotated clockwise,
#the opposite side (back) gets rotated counterclockwise, and the other sides get swapped in a 4-cycle
#new top is old left, new right is old top, new bottom is old right, new left is old bottom
#counterclockwise turn is just inverse

In [179]:
"""white is down, orange is left, green is back, yellow is up, red is right, and blue is front

now, the corners are as follows:
white-orange-green:0-4-8.
white-red-green:3-18-9
white-orange-blue:1-7-22
white-red-blue:2-17-23

yellow-orange-green:14-11-5
yellow-red-green:13-19-10
yellow-orange-blue:15-6-21
yellow-red-blue:12-16-20"""

'white is down, orange is left, green is back, yellow is up, red is right, and blue is front\n\nnow, the corners are as follows:\nwhite-orange-green:0-4-8.\nwhite-red-green:3-18-9\nwhite-orange-blue:1-7-22\nwhite-red-blue:2-17-23\n\nyellow-orange-green:14-11-5\nyellow-red-green:13-19-10\nyellow-orange-blue:15-6-21\nyellow-red-blue:12-16-20'

In [180]:
test_bleh=list(range(24))

In [181]:
old_bleh=test_bleh[16:20]

In [182]:
new_bleh=old_bleh[::-1]

In [183]:
rotate_counterclockwise(new_bleh)

[18, 17, 16, 19]

In [184]:
"""FIX THIS CLASS. ROTATIONS ARE PROBABLY INCONSISTENT"""
"""use clockwise for both side and its opposite, because rotate_clockwise spins counterclockwise for opposite side"""
"""now, figure out the rotations for the other sides"""
"""fixed?"""
class CubeRotate():
  def __init__(self,state=CANONICAL_CUBE):
    self.state=state
    self.faces={
      "DOWN":state[0:4],
      "LEFT":state[4:8],
      "BACK":state[8:12],
      "UP":state[12:16],
      "RIGHT":state[16:20],
      "FRONT":state[20:24]
    }
  def rotate_clockwise_back(self):
    new_face=deepcopy(self.faces)
    new_face["BACK"]=rotate_clockwise(self.faces["BACK"])
    new_face["FRONT"]=rotate_clockwise(self.faces["FRONT"])
    """new down is old left"""
    """"5 ends up where 0 is, 4 ends up where 3 is, 7 ends up where 2 is, 6 ends up where 1 is"""

    """this can be seen as a composition: first 4 ends up where 0 is,5 where 1 is, etc. then you rotate counterclockwise"""

    new_face["DOWN"]=self.faces["LEFT"]
    new_face["DOWN"]=rotate_counterclockwise(new_face["DOWN"])

    """new left is old up"""
    """12 ends up where 6 is, 13 ends up where 5 is, 14 ends up where 4 is, 15 ends up where 7 is"""
    """this can also be seen as a composition. first 12 ends up where 7 is, 13 ends up where 6 is, 14 ends up where 5 is, 15 ends up where 4 is."""
    """then you rotate counterclockwise."""


    new_face["LEFT"]=self.faces["UP"][::-1]
    new_face["LEFT"]=rotate_counterclockwise(new_face["LEFT"])

    """new up is old right"""
    """16 ends up where 15 is, 17 ends up where 12 is,18 ends up where 13 is, 19 ends up where 14 is"""
    """this can also be seen as a composition. first 16 ends up where 12 is, 17 where 13 is, 18 where 14 is, and 19 where 15 is"""
    """then you rotate CLOCKWISE. but because it's the UP side, you use rotate_counterclockwise"""

    new_face["UP"]=self.faces["RIGHT"]
    new_face["UP"]=rotate_counterclockwise(new_face["UP"])

    """new right is old down"""
    """0 ends up where 18 is,1 ends up where 17 is, 2 ends up where 16 is, 3 ends up where 19 is"""
    """this can also be seen as a composition: first 0 ends up where 19 is, 1 where 18 is, 2 where 17 is, 3 where 16 is"""
    """then you rotate CLOCKWISE. but because it's the RIGHT side, you use rotate_counterclockwise"""

    new_face["RIGHT"]=self.faces["DOWN"][::-1]
    new_face["RIGHT"]=rotate_counterclockwise(new_face["RIGHT"])

    new_state=new_face["DOWN"]+new_face["LEFT"]+new_face["BACK"]+new_face["UP"]+new_face["RIGHT"]+new_face["FRONT"]
    return CubeRotate(new_state)

  def rotate_clockwise_left(self):
    new_face=deepcopy(self.faces)

    new_face["LEFT"]=rotate_clockwise(self.faces["LEFT"])
    new_face["RIGHT"]=rotate_clockwise(self.faces["RIGHT"])

    """new down is old front"""
    """20 ends up where 2 is, 21 ends up where 1 is,22 ends up where 0 is, 23 ends up where 3 is"""
    """this can also be seen as a composition: first 20 ends up where 3 is, 21 ends up where 2 is, 22 ends up where 1 is, 23 ends up where 0 is"""
    """then you rotate counterclockwise"""

    new_face["DOWN"]=self.faces["FRONT"][::-1]
    new_face["DOWN"]=rotate_counterclockwise(new_face["DOWN"])

    """new front is old up"""
    """12 ends up where 23 is, 13 where 20 is, 14 where 21 is, 15 where 22 is"""
    """this can also be seen as a composition: first 12 ends up where 20 is, 13 where 21 is, 14 where 22 is, 15 where 23 is"""
    """then you rotate CLOCKWISE. but because it's the front side, you use rotate_counterclockwise"""

    new_face["FRONT"]=self.faces["UP"]
    new_face["FRONT"]=rotate_counterclockwise(new_face["FRONT"])

    """new up is old back"""
    """8 ends up where 14 is, 9 where 13 is, 10 where 12 is, 11 where 15 is"""
    """this can also be seen as a composition: first 8 ends up where 15 is, 9 where 14 is, 10 where 13 is and 11 where 12 is"""
    """then you rotate CLOCKWISE. but because it's the up side, you use rotate_counterclockwise"""

    new_face["UP"]=self.faces["BACK"][::-1]
    new_face["UP"]=rotate_counterclockwise(new_face["UP"])

    """new back is old down"""
    """0 ends up where 11 is, 1 where 8 is,2 where 9 is,3 where 10 is"""
    """this can also be seen as a composition: first 0 ends up where 8 is, 1 where 9 is, 2 where 10 is, 3 where 11 is"""
    """then you rotate counterclockwise"""

    new_face["BACK"]=self.faces["DOWN"]
    new_face["BACK"]=rotate_counterclockwise(new_face["BACK"])


    new_state=new_face["DOWN"]+new_face["LEFT"]+new_face["BACK"]+new_face["UP"]+new_face["RIGHT"]+new_face["FRONT"]
    return CubeRotate(new_state)
  def rotate_clockwise_down(self):
    """WORK IN PROGRESS"""

    new_face=deepcopy(self.faces)
    new_face["DOWN"]=rotate_clockwise(self.faces["DOWN"])
    new_face["UP"]=rotate_clockwise(self.faces["UP"])

    """new right is old front"""
    """20 ends up where 19 is, 21 where 16 is, 22 where 17 is, 23 where 18 is"""
    """this can also be seen as a composition: first 20 ends up where 16 is, 21 where 17 is, 22 where 18 is, 23 where 19 is"""
    """then you rotate CLOCKWISE. but because it's the right side, you use rotate_counterclockwise"""

    new_face["RIGHT"]=self.faces["FRONT"]
    new_face["RIGHT"]=rotate_counterclockwise(new_face["RIGHT"])

    """new front is old left"""
    """4 ends up where 22 is,5 where 21 is, 6 where 20 is, 7 where 23 is"""
    """this can also be seen as a composition: first 4 ends up where 23 is, 5 where 22 is, 6 where 21 is, 7 where 20 is"""
    """then you rotate CLOCKWISE. but because it's the front side, you use rotate_counterclockwise"""

    new_face["FRONT"]=self.faces["LEFT"][::-1]
    new_face["FRONT"]=rotate_counterclockwise(new_face["FRONT"])

    """new left is old back"""
    """8 ends up where 7 is, 9 where 4 is, 10 where 5 is, 11 where 6 is"""
    """this can also be seen as a composition: first 8 ends up where 4 is, 9 where 5 is, 10 where 6 is, 11 where 7 is"""
    """then you rotate counterclockwise"""

    new_face["LEFT"]=self.faces["BACK"]
    new_face["LEFT"]=rotate_counterclockwise(new_face["LEFT"])

    """new back is old right"""
    """16 ends up where 10 is, 17 where 9 is, 18 where 8 is, 19 where 11 is"""
    """this can also be seen as a composition: first 16 ends up where 11 is, 17 where 10 is, 18 where 9 is, 19 where 8 is"""
    """then you rotate counterclockwise"""

    new_face["BACK"]=self.faces["RIGHT"][::-1]
    new_face["BACK"]=rotate_counterclockwise(new_face["BACK"])

    new_state=new_face["DOWN"]+new_face["LEFT"]+new_face["BACK"]+new_face["UP"]+new_face["RIGHT"]+new_face["FRONT"]
    return CubeRotate(new_state)
  def adjacent_stickers_to_front(self):
    """NEED TO FIX THIS TOO"""
    """fixed?"""
    """NO, NOT FIXED!"""
    """make adjacent sticker indices manually?"""
    #clockwise
    l=[self.faces["LEFT"][3],self.faces["LEFT"][2]]
    u=[self.faces["UP"][3],self.faces["UP"][0]]
    r=[self.faces["RIGHT"][0],self.faces["RIGHT"][1]]
    d=[self.faces["DOWN"][2],self.faces["DOWN"][1]]
    return l+u+r+d

In [185]:
INDICES={"DOWN":list(range(0,4)),"LEFT":list(range(4,8)),"BACK":list(range(8,12)),"UP":list(range(12,16)),"RIGHT":list(range(16,20)),"FRONT":list(range(20,24))}

In [186]:
OLD_ADJACENT_STICKER_INDICES={"DOWN":CubeRotate().rotate_clockwise_left().rotate_clockwise_left().rotate_clockwise_left().adjacent_stickers_to_front(),
                          "UP":CubeRotate().rotate_clockwise_left().adjacent_stickers_to_front(),
                          "FRONT":CubeRotate().adjacent_stickers_to_front(),
                          "BACK":CubeRotate().rotate_clockwise_left().rotate_clockwise_left().adjacent_stickers_to_front(),
                          "LEFT":CubeRotate().rotate_clockwise_down().adjacent_stickers_to_front(),
                          "RIGHT":CubeRotate().rotate_clockwise_down().rotate_clockwise_down().rotate_clockwise_down().adjacent_stickers_to_front()
                          }

In [187]:
ADJACENT_STICKER_INDICES={"DOWN":[22,23,17,18,9,8,4,7],
                          "UP":[20,21,6,5,11,10,19,16],
                          "FRONT":[15,12,16,17,2,1,7,6],
                          "BACK":[5,4,0,3,18,19,13,14],
                          "LEFT":[1,0,8,11,14,15,21,22],
                          "RIGHT":[23,20,12,13,10,9,3,2]
}

In [188]:
cubetest=CubeRotate()
cubetest=cubetest.rotate_clockwise_back()
cubetest=cubetest.rotate_clockwise_left()
cubetest=cubetest.rotate_clockwise_back()

cubetest2=CubeRotate()
cubetest2=cubetest2.rotate_clockwise_left()
cubetest2=cubetest2.rotate_clockwise_back()
cubetest2=cubetest2.rotate_clockwise_left()
print(cubetest.state)
print(cubetest2.state)

[14, 13, 12, 15, 11, 8, 9, 10, 5, 6, 7, 4, 2, 1, 0, 3, 23, 20, 21, 22, 17, 18, 19, 16]
[14, 13, 12, 15, 11, 8, 9, 10, 5, 6, 7, 4, 2, 1, 0, 3, 23, 20, 21, 22, 17, 18, 19, 16]


In [189]:
def rotate_adjacent_stickers_clockwise(stickers):
  ls=list(range(24))
  for i,k in enumerate(stickers):
    ls[stickers[(i+2) %len(stickers)]]=k
  return ls

In [190]:
def full_rotate_side_clockwise(side):
  old_side_indices=INDICES[side]
  rotate=None
  if side in {"DOWN","BACK","LEFT"}:
    rotate=rotate_counterclockwise
  elif side in {"UP","FRONT","RIGHT"}:
    rotate=rotate_clockwise
  new_side_indices=rotate(old_side_indices)
  ls=ADJACENT_STICKER_INDICES[side]
  ls=rotate_adjacent_stickers_clockwise(ls)
  for i,k in enumerate(new_side_indices):
      ls[k]=old_side_indices[i]
  return ls

In [191]:
def inverse(permutation):
  ls=[0]*24
  for i,k in enumerate(permutation):
    ls[k]=i
  return ls

In [192]:
TURNS=[]
for i in SIDES:
  for j in DIRECTIONS:
    TURNS.append((i,j))

In [193]:
f_map={"CW":lambda side:full_rotate_side_clockwise(side),"CCW":lambda side:inverse(full_rotate_side_clockwise(side))}

In [194]:
PERMUTATIONS={turn:f_map[turn[1]](turn[0]) for turn in TURNS}

In [195]:
def apply_move(state, side, direction):
    perm_map = PERMUTATIONS[(side, direction)]
    new_state=[state[perm_map[i]] for i in range(24)]
    return tuple(new_state)

In [196]:
TURNS

[('DOWN', 'CW'),
 ('DOWN', 'CCW'),
 ('LEFT', 'CW'),
 ('LEFT', 'CCW'),
 ('BACK', 'CW'),
 ('BACK', 'CCW'),
 ('UP', 'CW'),
 ('UP', 'CCW'),
 ('RIGHT', 'CW'),
 ('RIGHT', 'CCW'),
 ('FRONT', 'CW'),
 ('FRONT', 'CCW')]

In [197]:
def shuffle(state,n=100):
  ls=[lambda st, side=side,direction=direction:apply_move(st,side,direction) for side,direction in TURNS]
  new_state=list(state)
  for i in range(n):
    act=random.choice(ls)
    #print(act)
    new_state=act(new_state)
    #print(new_state)
  return tuple(new_state)

In [198]:
def check(state,side):
  indices=INDICES[side]
  m=indices[0]
  return len(set(state[m:m+4]))==1

In [199]:
class ProblemaCubo(Problem):
  def actions(self,state):
    return TURNS
  def path_cost(self, c, state1, action, state2):
      """"Custo padrão"""
      return c + 1
  def result(self,state,action):
    return apply_move(state,action[0],action[1])
  def goal_test(self,state):
    for side in SIDES:
      if not check(state,side):
        return False
    return True


In [200]:
teste=shuffle(ESTADO_INICIAL)

In [201]:
teste2=ProblemaCubo(teste)

In [202]:
def h(node):
    total = 0
    for side in SIDES:
        indices = INDICES[side]
        m=indices[0]
        wrong=len(set(node.state[m:m+4]))-1
        total += wrong
    return (total + 3) // 4

In [203]:
"""def _generate_color_maps():
        #Generate valid color permutations (preserving opposite faces)
        # In standard cube: white-yellow, red-orange, blue-green are opposites
        opposites = {'w': 'y', 'y': 'w', 'r': 'o', 'o': 'r', 'b': 'g', 'g': 'b'}
        colors = ['w', 'y', 'r', 'o', 'b', 'g']

        valid_maps = []

        # Try to map each color to a canonical position
        # This is a simplified version - you can hardcode the 24 valid permutations
        # or generate them as shown below

        from itertools import permutations

        for perm in permutations(colors):
            # Check if permutation preserves opposite relationships
            valid = True
            color_map = {}
            for i, color in enumerate(colors):
                mapped = perm[i]
                color_map[color] = mapped

                # Check that opposites remain opposites
                opp_color = opposites[color]
                opp_mapped = opposites.get(mapped, None)

                if opp_mapped and perm[colors.index(opp_color)] != opp_mapped:
                    valid = False
                    break

            if valid:
                valid_maps.append(color_map)

        return valid_maps"""

"def _generate_color_maps():\n        #Generate valid color permutations (preserving opposite faces)\n        # In standard cube: white-yellow, red-orange, blue-green are opposites\n        opposites = {'w': 'y', 'y': 'w', 'r': 'o', 'o': 'r', 'b': 'g', 'g': 'b'}\n        colors = ['w', 'y', 'r', 'o', 'b', 'g']\n\n        valid_maps = []\n\n        # Try to map each color to a canonical position\n        # This is a simplified version - you can hardcode the 24 valid permutations\n        # or generate them as shown below\n\n        from itertools import permutations\n\n        for perm in permutations(colors):\n            # Check if permutation preserves opposite relationships\n            valid = True\n            color_map = {}\n            for i, color in enumerate(colors):\n                mapped = perm[i]\n                color_map[color] = mapped\n\n                # Check that opposites remain opposites\n                opp_color = opposites[color]\n                opp_mapped = 

In [204]:
#COLOR_MAP=_generate_color_maps()

In [205]:
"""def normalize(state):
        #Find canonical color representation
        if isinstance(state, list):
            state = tuple(state)

        canonical = None

        for color_map in COLOR_MAP:
            # Apply color mapping
            remapped = tuple(color_map.get(sticker) for sticker in state)

            # Keep smallest lexicographically
            if canonical is None or remapped < canonical:
                canonical = remapped

        return canonical"""

'def normalize(state):\n        #Find canonical color representation\n        if isinstance(state, list):\n            state = tuple(state)\n\n        canonical = None\n\n        for color_map in COLOR_MAP:\n            # Apply color mapping\n            remapped = tuple(color_map.get(sticker) for sticker in state)\n\n            # Keep smallest lexicographically\n            if canonical is None or remapped < canonical:\n                canonical = remapped\n\n        return canonical'

In [206]:
def normalize(state):
    """
    Rename colors based on their first occurrence order.
    This creates a canonical representation without trying all permutations.
    """
    if isinstance(state, list):
        state = tuple(state)

    # Map original colors to new names based on first appearance
    color_map = {}
    next_color = 0
    canonical = []

    for sticker in state:
        if sticker not in color_map:
            # Assign new color names in order of appearance
            # Using letters a,b,c,d,e,f for canonical form
            color_map[sticker] = chr(ord('a') + next_color)
            next_color += 1
        canonical.append(color_map[sticker])

    return tuple(canonical)

In [207]:
class NodeCubo(Node):
  def __init__(self, state, parent=None, action=None, path_cost=0):
        super().__init__(state, parent, action, path_cost)
        self._canonical = None
  def getCanonicalState(self):
    if self._canonical is None:
            self._canonical = normalize(self.state)
    return self._canonical
  def child_node(self, problem, action):
        """[Figure 3.10]"""
        next_state = problem.result(self.state, action)
        next_node = NodeCubo(next_state, self, action, problem.path_cost(self.path_cost, self.state, action, next_state))
        return next_node

In [208]:
def best_first_graph_search2(problem, f, display=False):
    """Search the nodes with the lowest f scores first.
    You specify the function f(node) that you want to minimize; for example,
    if f is a heuristic estimate to the goal, then we have greedy best
    first search; if f is node.depth then we have breadth-first search.
    There is a subtlety: the line "f = memoize(f, 'f')" means that the f
    values will be cached on the nodes as they are computed. So after doing
    a best first search you can examine the f values of the path returned."""
    #f = memoize(f, 'f')
    node = NodeCubo(problem.initial)
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    explored = set()
    frontier_set = set()
    canonical=node.getCanonicalState()
    frontier_set.add(canonical)
    frontier_set_map={}
    frontier_set_map[canonical]=node
    frontier_dict={}
    frontier_dict[node]=f(node)

    while frontier:
        node = frontier.pop()
        canonical = node.getCanonicalState()
        if canonical in frontier_set_map and frontier_set_map[canonical] is not node:
          continue
        """if canonical in frontier_set:
          frontier_set.remove(canonical)"""
        if problem.goal_test(node.state):
            if display:
                print(len(explored), "paths have been expanded and", len(frontier), "paths remain in the frontier")
            return node
        explored.add(canonical)
        frontier_dict[node]=f(node)
        for child in node.expand(problem):
          child_canonical=child.getCanonicalState()
          if child_canonical not in explored and child_canonical not in frontier_set:
              frontier.append(child)
              frontier_set.add(child_canonical)
              frontier_set_map[child_canonical]=child
              frontier_dict[child]=f(child)
          elif child_canonical in frontier_set:
              if f(child) < frontier_dict[frontier_set_map[child_canonical]]:
                  #substitute frontier[frontier_set_map[child_canonical]]
                  frontier.append(child)
                  frontier_set_map[child_canonical]=child
    return None

In [209]:
def best_first_graph_search2(problem, f, display=False):
    f = memoize(f, 'f')
    node = NodeCubo(problem.initial)
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    explored = set()

    # Track best f-value for each canonical state in frontier
    frontier_best = {}  # canonical -> best f-value
    frontier_node = {}  # canonical -> best node

    canonical = node.getCanonicalState()
    frontier_best[canonical] = f(node)
    frontier_node[canonical] = node

    while frontier:
        node = frontier.pop()
        canonical = node.getCanonicalState()
        node_f = f(node)

        # Skip if this node is outdated
        if canonical in frontier_best and frontier_best[canonical] < node_f:
            continue

        # Remove from frontier tracking
        if canonical in frontier_best:
            del frontier_best[canonical]
        if canonical in frontier_node:
            del frontier_node[canonical]

        if problem.goal_test(node.state):
            if display:
                print(len(explored), "paths expanded,", len(frontier), "paths remain")
            return node

        explored.add(canonical)

        for child in node.expand(problem):
            child_canonical = child.getCanonicalState()
            child_f = f(child)

            if child_canonical in explored:
                continue

            if child_canonical in frontier_best:
                # If this path is better, update
                if child_f < frontier_best[child_canonical]:
                    frontier_best[child_canonical] = child_f
                    frontier_node[child_canonical] = child
                    frontier.append(child)  # Add new node (old will be skipped)
            else:
                # New state
                frontier_best[child_canonical] = child_f
                frontier_node[child_canonical] = child
                frontier.append(child)

    return None

In [210]:
def best_first_graph_search3(problem, h, display=False):
    frontier_h = {}  # canonical -> h-value cache

    def f_with_cache(node):
        canonical = node.getCanonicalState()
        if canonical in frontier_h:
            h_val = frontier_h[canonical]
        else:
            h_val = h(node)
            frontier_h[canonical] = h_val
        return node.path_cost + h_val

    node = NodeCubo(problem.initial)
    frontier = PriorityQueue('min', f_with_cache)
    frontier.append(node)
    explored = set()

    # Track best f-value for each canonical state in frontier
    frontier_best = {}  # canonical -> best f-value (using f_with_cache)
    frontier_node = {}  # canonical -> best node

    canonical = node.getCanonicalState()
    node_f = f_with_cache(node)  # Use f_with_cache consistently
    frontier_best[canonical] = node_f
    frontier_node[canonical] = node
    frontier_h[canonical] = h(node)  # Still cache h-value

    while frontier:
        node = frontier.pop()
        canonical = node.getCanonicalState()
        node_f = f_with_cache(node)

        # Skip if this node is outdated
        if canonical in frontier_best and frontier_best[canonical] < node_f:
            continue

        # Remove from frontier tracking
        if canonical in frontier_best:
            del frontier_best[canonical]
        if canonical in frontier_node:
            del frontier_node[canonical]

        if problem.goal_test(node.state):
            if display:
                print(len(explored), "paths expanded,", len(frontier), "paths remain")
            return node

        explored.add(canonical)

        for child in node.expand(problem):
            child_canonical = child.getCanonicalState()
            child_f = f_with_cache(child)

            if child_canonical in explored:
                continue

            if child_canonical in frontier_best:
                if child_f < frontier_best[child_canonical]:
                    frontier_best[child_canonical] = child_f
                    frontier_node[child_canonical] = child
                    frontier.append(child)
            else:
                frontier_best[child_canonical] = child_f
                frontier_node[child_canonical] = child
                frontier.append(child)

    return None

In [211]:
def astar_search2(problem, h=None, display=False):
    """A* search is best-first graph search with f(n) = g(n)+h(n).
    You need to specify the h function when you call astar_search, or
    else in your Problem subclass."""
    #h = memoize(h or problem.h, 'h')
    return best_first_graph_search3(problem,h, display)

In [212]:
#teste3=astar_search2(teste2,h=h,display=True)

In [213]:
teste=shuffle(ESTADO_INICIAL,n=10)

In [214]:
teste=apply_move(apply_move(apply_move(apply_move(apply_move(apply_move(ESTADO_INICIAL,"RIGHT","CW"),"FRONT","CW"),"DOWN","CW"),"UP","CW"),"BACK","CCW"),"FRONT","CW")

In [215]:
teste2=ProblemaCubo(teste)

In [216]:
def breadth_first_graph_search2(problem):
    """[Figure 3.11]
    Note that this function can be implemented in a
    single line as below:
    return graph_search(problem, FIFOQueue())
    """
    node = NodeCubo(problem.initial)
    if problem.goal_test(node.state):
        return node
    frontier = deque([node])
    explored = set()
    while frontier:
        node = frontier.popleft()
        explored.add(node.state)
        for child in node.expand(problem):
            if child.getCanonicalState() not in explored and child not in frontier:
                if problem.goal_test(child.state):
                    return child
                frontier.append(child)
    return None

In [217]:
teste4=astar_search2(teste2,h=h,display=True)

23 paths expanded, 197 paths remain


In [218]:
testando2=shuffle(ESTADO_INICIAL,n=12)

In [219]:
teste2=ProblemaCubo(testando2)

In [220]:
teste6=astar_search2(teste2,h=h,display=True)

460295 paths expanded, 1184777 paths remain


In [221]:
testando3=shuffle(ESTADO_INICIAL,n=8)

In [222]:
teste2=ProblemaCubo(testando3)

In [223]:
teste4=astar_search(teste2,h=h,display=True)

658 paths have been expanded and 4463 paths remain in the frontier


In [224]:
teste6=astar_search2(teste2,h=h,display=True)

621 paths expanded, 3978 paths remain


In [225]:
teste3=breadth_first_graph_search2(teste2)

KeyboardInterrupt: 

In [226]:
ROTATIONS_1=[
    lambda cube:cube,
    lambda cube:cube.rotate_clockwise_left(),
    lambda cube:cube.rotate_clockwise_left().rotate_clockwise_left(),
    lambda cube:cube.rotate_clockwise_left().rotate_clockwise_left().rotate_clockwise_left(),
    lambda cube:cube.rotate_clockwise_down(),
    lambda cube:cube.rotate_clockwise_down().rotate_clockwise_down().rotate_clockwise_down()
]
ROTATIONS_2=[lambda cube:cube,
             lambda cube:cube.rotate_clockwise_back(),
          lambda cube:cube.rotate_clockwise_back().rotate_clockwise_back(),
          lambda cube:cube.rotate_clockwise_back().rotate_clockwise_back().rotate_clockwise_back()]

In [227]:
ROTATIONS=[]
for i in ROTATIONS_1:
  for j in ROTATIONS_2:
    ROTATIONS.append(lambda cube, i=i,j=j :j(i(cube)))

In [228]:
ROTATION_PERMUTATIONS={i: ROTATIONS[i](CubeRotate()).state for i in range(24)}

In [229]:
for i in range(24):
  print(ROTATIONS[i](CubeRotate()).state[0],ROTATIONS[i](CubeRotate()).state[4],ROTATIONS[i](CubeRotate()).state[8])

0 4 8
5 14 11
13 19 10
18 3 9
22 7 1
4 8 0
9 18 3
17 23 2
15 6 21
7 1 22
2 17 23
16 12 20
11 5 14
6 21 15
20 16 12
19 10 13
3 9 18
10 13 19
12 20 16
23 2 17
1 22 7
21 15 6
14 11 5
8 0 4


In [230]:
def get_inverse_rotation(rot_idx):
    """Find the rotation that undoes rot_idx."""
    # Apply rot_idx to canonical cube
    cube = ROTATIONS[rot_idx](CubeRotate())

    # Now find which rotation brings this back to canonical
    # This is brute force but only done once at startup
    for j in range(24):
        test_cube = ROTATIONS[j](cube)
        if (test_cube.state==CubeRotate().state):
            return j
    return 0

In [231]:
INVERSE_ROTATION = {}
for i in range(24):
    INVERSE_ROTATION[i] = get_inverse_rotation(i)

In [232]:
INVERSE_ROTATION

{0: 0,
 1: 3,
 2: 2,
 3: 1,
 4: 12,
 5: 23,
 6: 6,
 7: 17,
 8: 8,
 9: 9,
 10: 10,
 11: 11,
 12: 4,
 13: 19,
 14: 14,
 15: 21,
 16: 20,
 17: 7,
 18: 18,
 19: 13,
 20: 16,
 21: 15,
 22: 22,
 23: 5}

In [233]:
ROTATIONS_TEST = set()
for x in range(4):
    for y in range(4):
        for z in range(4):
            cube = CubeRotate()
            for _ in range(x): cube = cube.rotate_clockwise_left()
            for _ in range(y): cube = cube.rotate_clockwise_down()
            for _ in range(z): cube = cube.rotate_clockwise_back()
            ROTATIONS_TEST.add(tuple(cube.state))

print(f"Unique rotations: {len(ROTATIONS_TEST)}")

Unique rotations: 24


In [234]:
ROTATIONS_TEST=list(ROTATIONS_TEST)

In [235]:
def apply_rotation_to_state(state, rot_idx):
    """Apply rotation rot_idx to a state (list or tuple of 24 items)."""
    perm = ROTATION_PERMUTATIONS[rot_idx]
    return tuple(state[perm[i]] for i in range(24))

In [236]:
# Mapping 1: From rotation index to the corner it produces on a solved color-canonical cube
rot_idx_to_corner = {}

solved=ESTADO_INICIAL  #solved state

for i in range(24):
    rotated = apply_rotation_to_state(solved, i)
    corner = (rotated[0], rotated[4], rotated[8])
    rot_idx_to_corner[i] = corner

# Mapping 2: From corner to the rotation index that PRODUCED it
corner_to_rot_idx = {corner: i for i, corner in rot_idx_to_corner.items()}

# Mapping 3: From corner to the INVERSE rotation (to canonicalize)
corner_to_canonical_rotation = {}
for corner, rot_idx in corner_to_rot_idx.items():
    corner_to_canonical_rotation[corner] = INVERSE_ROTATION[rot_idx]
    #print(tuple(sorted(corner)))

In [237]:
corner_to_canonical_rotation

{('w', 'o', 'g'): 0,
 ('o', 'y', 'g'): 3,
 ('y', 'r', 'g'): 2,
 ('r', 'w', 'g'): 1,
 ('b', 'o', 'w'): 12,
 ('o', 'g', 'w'): 23,
 ('g', 'r', 'w'): 6,
 ('r', 'b', 'w'): 17,
 ('y', 'o', 'b'): 8,
 ('o', 'w', 'b'): 9,
 ('w', 'r', 'b'): 10,
 ('r', 'y', 'b'): 11,
 ('g', 'o', 'y'): 4,
 ('o', 'b', 'y'): 19,
 ('b', 'r', 'y'): 14,
 ('r', 'g', 'y'): 21,
 ('w', 'g', 'r'): 20,
 ('g', 'y', 'r'): 7,
 ('y', 'b', 'r'): 18,
 ('b', 'w', 'r'): 13,
 ('w', 'b', 'o'): 16,
 ('b', 'y', 'o'): 15,
 ('y', 'g', 'o'): 22,
 ('g', 'w', 'o'): 5}

In [238]:
def normalize_rotation(state):
    """
    Take a tate and return its rotation-canonical form.
    """
    # Extract the first corner
    corner = (state[0],
              state[4],
              state[8])
    # Find which rotation brings this corner to canonical position
    rot_idx = corner_to_canonical_rotation.get(corner)
    if rot_idx is None:
        print(f"Corner {corner} not found!")
        return state

    # Apply that rotation
    rotation_perm = ROTATION_PERMUTATIONS[rot_idx]
    rotated = tuple(state[rotation_perm[i]] for i in range(24))

    return rotated

In [239]:
cube_test=shuffle(ESTADO_INICIAL,n=30)
normalize(normalize_rotation(cube_test))==normalize(cube_test)

False

In [283]:
est_test=shuffle(ESTADO_INICIAL,n=10)
for j in range(24):
  print(normalize_rotation(est_test))
  print(normalize_rotation(tuple(est_test[ROTATION_PERMUTATIONS[j][i]] for i in range(24))))

('b', 'b', 'g', 'o', 'y', 'o', 'g', 'w', 'o', 'g', 'b', 'g', 'y', 'w', 'y', 'r', 'r', 'r', 'w', 'r', 'b', 'y', 'o', 'w')
('b', 'b', 'g', 'o', 'y', 'o', 'g', 'w', 'o', 'g', 'b', 'g', 'y', 'w', 'y', 'r', 'r', 'r', 'w', 'r', 'b', 'y', 'o', 'w')
('b', 'b', 'g', 'o', 'y', 'o', 'g', 'w', 'o', 'g', 'b', 'g', 'y', 'w', 'y', 'r', 'r', 'r', 'w', 'r', 'b', 'y', 'o', 'w')
('r', 'y', 'w', 'y', 'g', 'w', 'y', 'o', 'y', 'b', 'w', 'o', 'o', 'g', 'b', 'b', 'w', 'r', 'r', 'r', 'g', 'o', 'g', 'b')
('b', 'b', 'g', 'o', 'y', 'o', 'g', 'w', 'o', 'g', 'b', 'g', 'y', 'w', 'y', 'r', 'r', 'r', 'w', 'r', 'b', 'y', 'o', 'w')
('w', 'r', 'r', 'r', 'o', 'b', 'b', 'g', 'g', 'b', 'g', 'o', 'g', 'o', 'y', 'w', 'r', 'y', 'w', 'y', 'y', 'o', 'w', 'b')
('b', 'b', 'g', 'o', 'y', 'o', 'g', 'w', 'o', 'g', 'b', 'g', 'y', 'w', 'y', 'r', 'r', 'r', 'w', 'r', 'b', 'y', 'o', 'w')
('r', 'w', 'r', 'r', 'b', 'g', 'o', 'g', 'w', 'y', 'r', 'y', 'w', 'g', 'o', 'y', 'o', 'w', 'b', 'y', 'b', 'b', 'o', 'g')
('b', 'b', 'g', 'o', 'y', 'o', '

In [270]:
class NodeCubo(Node):
  def __init__(self, state, parent=None, action=None, path_cost=0):
        super().__init__(state, parent, action, path_cost)
        self._canonical = None
  def getCanonicalState(self):
    if self._canonical is None:
            self._canonical = normalize_rotation(self.state)
    return self._canonical
  def child_node(self, problem, action):
        """[Figure 3.10]"""
        next_state = problem.result(self.state, action)
        next_node = NodeCubo(next_state, self, action, problem.path_cost(self.path_cost, self.state, action, next_state))
        return next_node

In [272]:
teste_blah=shuffle(ESTADO_INICIAL,n=30)

In [275]:
teste_2=ProblemaCubo(teste_blah)

In [276]:
teste7=astar_search2(teste_2,h=h,display=True)

324889 paths expanded, 1304366 paths remain


In [251]:
test_test=apply_move(ESTADO_INICIAL,"DOWN","CCW")

In [252]:
test_test[0],test_test[4],test_test[8]

('w', 'b', 'o')

In [ ]:
""""R' U' R' F' U F U' R' F R F' U' R U2 R"""#identity sequence, test to see if turns are right

In [253]:
test_test2=apply_move(ESTADO_INICIAL,"RIGHT","CCW")
test_test2=apply_move(test_test2,"UP","CCW")
test_test2=apply_move(test_test2,"RIGHT","CCW")
test_test2=apply_move(test_test2,"FRONT","CCW")
test_test2=apply_move(test_test2,"UP","CW")
test_test2=apply_move(test_test2,"FRONT","CW")
test_test2=apply_move(test_test2,"UP","CCW")
test_test2=apply_move(test_test2,"RIGHT","CCW")
test_test2=apply_move(test_test2,"FRONT","CW")
test_test2=apply_move(test_test2,"RIGHT","CW")
test_test2=apply_move(test_test2,"FRONT","CCW")
test_test2=apply_move(test_test2,"UP","CCW")
test_test2=apply_move(test_test2,"RIGHT","CW")
test_test2=apply_move(test_test2,"UP","CCW")
test_test2=apply_move(test_test2,"UP","CCW")
test_test2=apply_move(test_test2,"RIGHT","CW")

In [258]:
print(test_test2)
print(ESTADO_INICIAL)

('w', 'w', 'w', 'w', 'o', 'o', 'o', 'o', 'g', 'g', 'g', 'g', 'y', 'y', 'y', 'y', 'r', 'r', 'r', 'r', 'b', 'b', 'b', 'b')
['w', 'w', 'w', 'w', 'o', 'o', 'o', 'o', 'g', 'g', 'g', 'g', 'y', 'y', 'y', 'y', 'r', 'r', 'r', 'r', 'b', 'b', 'b', 'b']


In [261]:
test_test3=CubeRotate()
test_set={tuple(test_test3.state)}
for i in range(1000):
  f=random.choice([CubeRotate.rotate_clockwise_back,CubeRotate.rotate_clockwise_down,CubeRotate.rotate_clockwise_left])
  test_test3=f(test_test3)
  test_set.add(tuple(test_test3.state))

In [263]:
len(test_set)

24

In [284]:
#testing

In [285]:
def canonical_fast(state):

    best = None

    # Step 1:
    # Find minimal first-corner representation

    best_corner = None
    candidates = []

    for r in range(24):

        perm = ROTATION_PERMUTATIONS[r]

        c = (
            state[perm[0]],
            state[perm[4]],
            state[perm[8]]
        )

        if best_corner is None or c < best_corner:

            best_corner = c
            candidates = [r]

        elif c == best_corner:

            candidates.append(r)

    # Step 2:
    # Only test candidate rotations

    for r in candidates:

        perm = ROTATION_PERMUTATIONS[r]

        rotated = tuple(
            state[perm[i]]
            for i in range(24)
        )

        if best is None or rotated < best:
            best = rotated

    return best

In [311]:
class NodeCubo(Node):
  def __init__(self, state, parent=None, action=None, path_cost=0):
        super().__init__(state, parent, action, path_cost)
        self._canonical = None
  def getCanonicalState(self):
    if self._canonical is None:
            self._canonical = canonical_fast(self.state)
    return self._canonical
  def child_node(self, problem, action):
        """[Figure 3.10]"""
        next_state = problem.result(self.state, action)
        next_node = NodeCubo(next_state, self, action, problem.path_cost(self.path_cost, self.state, action, next_state))
        return next_node

In [310]:
teste7=astar_search2(teste_2,h=h,display=True)

81249 paths expanded, 223828 paths remain
